# PD Model Development: Baseline Logistic Regression
### Flagship PD Model Project · Stage 2 (Modelling)

> **Objective.** Fit a baseline logistic regression on the WOE-encoded features from the driver scan, check that it behaves the way credit logic says it should, and record its discrimination on the held-out test set as the benchmark for later iterations.

| Field | Detail |
|---|---|
| **Project** | Flagship PD Model - Project 2: PD Model Development |
| **Stage** | Stage 2 - Baseline Model |
| **Dataset** | Home Credit Default Risk - WOE-encoded `train_woe` / `test_woe` (from `01_Default_Rate_Driver_Scan_EDA`) |
| **Target** | `TARGET` (1 = default / serious delinquency, 0 = repaid) |
| **Author** | Chuch |
| **AI assistance** | Built with AI assistance (Claude): notebook scaffold, cells marked "Written by AI", and editing of the write-ups. Predictions and final decisions are the author's. |
| **Date started / last updated** | *<YYYY-MM-DD>* / *<YYYY-MM-DD>* |
| **Notebook version** | *<v0.1 - draft>* |
| **Status** | In Progress |

**Reading guide - the E / O / D convention.** Each analysis block is written as three linked notes:
- **Explanation** - the hypothesis / business rationale *before* looking at the output.
- **Observation** - what the output actually showed.
- **Decision** - the action taken and its justification.

**Reproducibility.** *(placeholder - Python version, key library versions, `RANDOM_STATE`, the input file snapshot date, and any manual steps needed to re-run top-to-bottom.)*

## Contents

- **Project Setup** - environment & imports, display settings, style guide, data loading
- **1. Scope & Inputs** - target definition and success criteria · inherited inputs check
- **2. Feature Set Check** - collinearity between WOE features · baseline feature list
- **3. Baseline Logistic Regression** - model setup · fit · coefficient sign check
- **4. Baseline Results** - discrimination (AUC / Gini, KS) · train vs test gap · rank ordering
- **5. Findings Summary**
- **6. Assumptions, Limitations & Data Lineage**
- **7. Next Steps** - what the next iteration picks up (validation, scaling)

# Project Setup

## Environment and Imports

Imports, repo paths and the project's own utilities (`python_style_util`, `evaluation_utils`, `credit_utils`). `credit_utils` is auto-reloaded, so edits to its modules show up without restarting the kernel.

In [ ]:
# Standard library
import os
import sys
import json
import pickle
import warnings
import joblib
import shutil

# Data
import numpy as np
import pandas as pd
import json

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker

# Modelling
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.stats import ks_2samp
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.api import add_constant 
import sklearn


# Jupyter
from IPython.core.interactiveshell import InteractiveShell
from IPython.display import display


# Repo paths
def find_repo_root(start, repo_name='CreditRiskLearning'):
    path = os.path.abspath(start)
    while True:
        if os.path.basename(path) == repo_name:
            return path
        parent = os.path.dirname(path)
        if parent == path:
            raise FileNotFoundError(f"Could not find repo root: {repo_name}")
        path = parent


repo_root = find_repo_root(os.getcwd())
sys.path.append(os.path.join(repo_root, 'Core Resources'))
sys.path.append(os.path.join(repo_root, 'Core Resources', 'Scripts'))

# Style utils
from python_style_util import (
    palette, fonts, sizes,
    p, f, s,
    C, Shade, Size,
    set_style,
    gradient, traffic_light, diverging, diverging_cmap, risk_cmap, RISK_LOW, RISK_HIGH,
    multi_series, binary_palette,
    chart_title, annotate, style_df,
    show_palette, show_fonts, show_tokens, show_colormaps, show_style_guide,
)
import evaluation_utils as eu

# Project toolkit (credit_utils)
%load_ext autoreload
%autoreload 2

from credit_utils import eda, quality, woe, stats, style

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Display Settings

Pandas, NumPy and Matplotlib display options, plus the project chart style.

In [ ]:
# Pandas
pd.set_option('display.float_format', '{:.2f}'.format)  # 2 decimal places on floats
pd.set_option('display.max_columns', None)              # show all columns
pd.set_option('display.max_rows', 100)                  # show up to 100 rows
pd.set_option('display.width', None)                    # don't wrap wide dataframes
pd.set_option('display.colheader_justify', 'left')      # left-align column headers
pd.set_option('display.precision', 4)                   # decimal precision for describe()
pd.set_option('display.large_repr', 'truncate')         # truncate instead of summary for large dfs

# NumPy
np.set_printoptions(
    precision=4,        # decimal places
    suppress=True,      # no scientific notation
    linewidth=120,      # wrap width
    threshold=1000,     # show up to 1000 elements before summarising with ...
    edgeitems=5,        # show 5 items at each end when it does summarise
)

# Matplotlib
%matplotlib inline
plt.rcParams['figure.dpi'] = 120             # sharper figures
plt.rcParams['savefig.dpi'] = 150            # higher res when saving
plt.rcParams['savefig.bbox'] = 'tight'       # no clipped labels when saving
plt.rcParams['savefig.facecolor'] = 'white'  # white background on saved figures

# Jupyter
InteractiveShell.ast_node_interactivity = 'all'  # print every expression, not just the last

# Warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=ConvergenceWarning)

# Style
set_style()

## Style Guide

Quick visual reference for the project's colour tokens and fonts.

In [ ]:
# Colour tokens and fonts
show_tokens()
show_fonts()

## Data Loading

Loads the handoff from the driver scan: the WOE-encoded train/test frames, the fitted `BinningProcess` (needed later to score new data - reuse, never refit), the `ORG_BAND` lookup and the metadata file with the decisions carried over.

In [ ]:
# Handoff folder from the driver scan
INPUT_DIR = os.path.join(repo_root, 'Projects', 'Flagship PD Model Project',
                         '02_PD_Model_Development', 'inputs')

# WOE-encoded frames
train = pd.read_parquet(os.path.join(INPUT_DIR, 'train_woe.parquet'))
test = pd.read_parquet(os.path.join(INPUT_DIR, 'test_woe.parquet'))

# Fitted binning process and ORG_BAND lookup
with open(os.path.join(INPUT_DIR, 'binning_process.pkl'), 'rb') as fh:
    bp = pickle.load(fh)
with open(os.path.join(INPUT_DIR, 'org_band_map.json')) as fh:
    org_band_map = json.load(fh)

# Carried-over decisions
with open(os.path.join(INPUT_DIR, 'metadata.json')) as fh:
    metadata = json.load(fh)

model_features = metadata['model_features']

# Features / target
X_train, y_train = train[model_features], train['TARGET']
X_test, y_test = test[model_features], test['TARGET']

print(f"Train: {X_train.shape[0]:,} rows × {X_train.shape[1]} features")
print(f"Test:  {X_test.shape[0]:,} rows × {X_test.shape[1]} features")

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.

# Project Body

## 1. Scope & Inputs

### 1.1 Target Definition & Success Criteria

- **Target:** `TARGET` = 1 if the client had payment difficulties (late payment over X days on at least one of the first Y instalments). A payment-difficulty proxy, not a full 90+ DPD default.
- **Population:** Home Credit application data, application-time features only (the six behavioural tables are not used).
- **Model scope:** baseline only - one logistic regression on the WOE features, no tuning.
- **Success criteria:** *(set the bar before fitting - e.g. test Gini above X, train/test Gini gap under Y, all coefficient signs as expected)*

#### 1.1a Minimum test Gini: 0.45

The floor is 0.36 (EXT_SOURCE_3 alone). The +0.09 margin is moderate because the three EXT scores overlap with each other and with age, and several features share loan-amount and employment information. The tuning notebook will set its own bar, anchored on this baseline's actual test Gini.

#### 1.1b Maximum train/test Gini gap: 0.02

Logistic regression on 14 WOE-binned features with about 246k training rows has very little capacity to overfit. A small positive gap is expected because the bins were fitted on train and test Gini carries about ±0.01 sampling noise, but anything above 0.02 would point to overfitting or leakage.

#### 1.1c Coefficient signs: all negative, none significantly positive

In our optbinning WOE (ln of %non-event over %event), high-risk bins have negative WOE, so every coefficient should be negative for riskier bins to raise PD. A coefficient is a hard fail if it is **significantly positive**: its 95% confidence interval (statsmodels Logit, same specification as the sklearn fit) lies entirely above zero. That means applicants whose features disagree (for example a young applicant with a strong external score) are scored in a direction the data contradicts, and as the baseline it would give 03's challengers an unreliable benchmark. A positive coefficient whose interval includes zero is not distinguishable from zero, and is investigated and noted rather than failed. Fix: drop the offending feature (or, if the cause is a collinear pair, the lower-IV member) and refit.

*Amended after the first fit.* The original rule failed any positive sign. DAYS_BIRTH came out at +0.064 (95% CI 0.000 to 0.127, p = 0.049) with every VIF below 2, so it still fails under the amended rule and was dropped (see 2.2).

#### 1.1d Rank ordering: strictly monotonic across test PD deciles (zero reversals)

Each test decile holds about 6,150 rows, with defaults ranging from roughly 100 in the safest decile to over 1,500 in the riskiest. At a Gini near 0.45, adjacent deciles are expected to differ by more than twice their sampling noise, so chance reversals should be rare (about 1% in a simulation using assumed decile rates). Any reversal is treated as a real ranking weakness in that score band and investigated in 4.3.

### 1.2 Inherited Inputs Check

**Expectation**: train 246,005 rows and test 61,502 rows (sklearn rounds the 20% test size up), summing to 307,507 (307,511 less the 4 CODE_GENDER = 'XNA' rows dropped in the driver scan). Default rate about 8.07% in both, because the split was stratified on TARGET. Stratification fixes the default rate across splits so that any train/test difference in Gini or calibration reflects the model, not the split. It does not guarantee other feature distributions match. At this sample size an unstratified split would differ by only about ±0.1 points, so the benefit here is small but free.

In [ ]:
# Drop Features before modelling based on Section 2 Results



In [ ]:
# Train + Test Length Check - should equal 307507 - not 307511
# 4 Rows dropped that contained CODE_GENDER == 'XNA'

print(len(train) + len(test))

In [ ]:
# Check Stratified Split

print(f"{train['TARGET'].mean().round(4) * 100}%")

print(f"{test['TARGET'].mean().round(4) * 100}%")

In [ ]:
# Feature Column Check 

for c in metadata['model_features']:
    
    if c not in train.columns:
    
        print(c)
        

for c in metadata['model_features']:
    
    if c not in test.columns:
    
        print(c)
        
for c in train.columns:
    
    if c not in metadata['model_features']:
    
        print(c)
        

for c in test.columns:
    
    if c not in metadata['model_features']:
    
        print(c)        

In [ ]:
# Missingness Check

train.isna().sum()

test.isna().sum()

**Observation.** Train 246,005 rows and test 61,502 rows, summing to 307,507 (4 rows with CODE_GENDER = 'XNA' dropped in the driver scan before the split). Default rate 8.07% in both, confirming the stratified split. All 14 model features are present in both frames, and the only other column is TARGET (see cell 23 output), so no IDs, protected attributes or derived target columns came through. No missing values in any WOE column.

**Decision.** Inputs reconcile with the driver scan handoff on row counts, default rate and feature list, so they are used as-is. Proceed to Section 2 to check collinearity before fitting.

## 2. Feature Set Check

**Purpose.** The driver scan chose the 14 features on IV. Before fitting, check that none of them are so correlated that the coefficients become unstable. Candidate pairs to watch are listed in the Section 5 exclusions of the driver scan.

### 2.1 Correlation & VIF on WOE Features

**Explanation.** Correlation checks overlap between two features at a time; VIF checks how well each feature can be rebuilt from all 13 others combined (VIF = 1 / (1 - R squared)), which catches group overlap that pairwise correlation misses.

- **Thresholds.** VIF above 5 (R squared above 0.8) or pairwise correlation above 0.8 triggers a drop. The bar is strict because a flipped coefficient sign is a hard fail (1.1c), WOE encoding usually keeps VIF low so anything above 5 is a real signal, and the features are IV-ranked so dropping the weaker one of an overlapping group costs little.
- **Process.** Drop one feature at a time: remove the lower-IV feature from the worst group, recompute VIF, and repeat, because removing one feature lowers its partners' VIF.
- **Prediction.** Highest pair: DAYS_EMPLOYED vs EMPLOYED_AGE, because EMPLOYED_AGE is derived from DAYS_EMPLOYED (DAYS_EMPLOYED / DAYS_BIRTH) and both share the same pensioner / unemployed NaN bin (about 18% of rows). Highest VIF: DAYS_EMPLOYED, because the remaining features combined can largely rebuild it, with DAYS_BIRTH and EMPLOYED_AGE still in the feature set.

In [ ]:
shape = X_train.corr().shape

mask = np.triu(np.ones(shape), k=1).astype(bool)

corr = X_train.corr().where(mask)

X_train.corr().where(mask).stack().abs().sort_values(ascending=False).head(20)

In [ ]:
fig,ax = plt.subplots(figsize=(10,8))

ax = sns.heatmap(corr)

In [ ]:
# Written by AI
# VIF on all 14 features (train only), before any drop
def vif_table(X):
    X_const = add_constant(X)
    vif = pd.DataFrame({
        'feature': X_const.columns,
        'VIF': [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])],
    })
    return vif[vif['feature'] != 'const'].sort_values('VIF', ascending=False).reset_index(drop=True)

vif_table(X_train)

In [ ]:
# Written by AI
# Test the drop: EMPLOYED_AGE removed (lower IV than DAYS_EMPLOYED); X_train itself is unchanged
vif_table(X_train.drop(columns='EMPLOYED_AGE'))

**Observation.** Both predictions held. DAYS_EMPLOYED vs EMPLOYED_AGE is the highest pair at r = 0.94, the only pair above the 0.8 threshold. With all 14 features, DAYS_EMPLOYED has the highest VIF (12.89), with EMPLOYED_AGE close behind (12.14); every other feature is below 2.3. After dropping EMPLOYED_AGE, DAYS_EMPLOYED falls from 12.89 to 1.13 and DAYS_BIRTH from 2.26 to 1.66, and the highest remaining VIF is NAME_INCOME_TYPE at 1.97. ORG_BAND vs NAME_INCOME_TYPE (r = 0.65, VIFs 1.89 and 1.97) is the next largest overlap, likely the pensioner cluster, but sits below both thresholds.

**Decision.** Drop EMPLOYED_AGE: it is the lower-IV feature of the only pair above threshold and is derived from DAYS_EMPLOYED and DAYS_BIRTH, both of which stay in. One drop brings every VIF below 2, so no further drops are needed. Keep ORG_BAND and NAME_INCOME_TYPE, as both are below threshold; check their coefficient signs in 3.3. Baseline goes forward with 13 features.

### 2.2 Baseline Feature List

**Decision.** 12 features go into the baseline. Two drops: EMPLOYED_AGE (collinearity, see 2.1) and DAYS_BIRTH (wrong-signed coefficient in the first fit, see 3.3).

| Feature | Kept / Dropped | Reason |
|---|---|---|
| EXT_SOURCE_3 | Kept | Below both thresholds |
| EXT_SOURCE_2 | Kept | Below both thresholds |
| EXT_SOURCE_1 | Kept | Below both thresholds |
| DAYS_EMPLOYED | Kept | VIF 12.89 falls to 1.13 once EMPLOYED_AGE is dropped |
| ANNUITY_CREDIT | Kept | Below both thresholds |
| EMPLOYED_AGE | Dropped | r = 0.94 with DAYS_EMPLOYED, VIF 12.14; lower IV and derived from DAYS_EMPLOYED / DAYS_BIRTH |
| AMT_GOODS_PRICE | Kept | Below both thresholds |
| DAYS_BIRTH | Dropped | Below both VIF thresholds, but its coefficient was positive (+0.06, p = 0.049) in the first fit, a flipped sign under 1.1c |
| OCCUPATION_TYPE | Kept | Below both thresholds |
| CREDIT_GOODS | Kept | Below both thresholds |
| ORG_BAND | Kept | r = 0.65 with each other, below threshold; watch signs in 3.3 |
| NAME_INCOME_TYPE | Kept | r = 0.65 with each other, below threshold; watch signs in 3.3 |
| REGION_RATING_CLIENT_W_CITY | Kept | Below both thresholds |
| NAME_EDUCATION_TYPE | Kept | Below both thresholds |

In [ ]:
# Exclusion List

exclusion_list = ['EMPLOYED_AGE', 'DAYS_BIRTH']

model_features

# Baseline feature list; rebuild train and test from the same list so the columns match
baseline_features = [f for f in model_features if f not in exclusion_list]

X_train, X_test = train[baseline_features], test[baseline_features]

print(f"Baseline features: {len(baseline_features)}")
print(f"Train: {X_train.shape}  Test: {X_test.shape}")

## 3. Baseline Logistic Regression

### 3.1 Model Setup

**Class weighting:** None. The data's 8.07% default rate is kept as is. With about 19,800 defaults in train, the minority class is well represented, so reweighting isn't needed to learn it. 'balanced' would multiply every applicant's odds of default by about 11.4, leaving rank ordering (Gini) unchanged but pushing the average predicted PD far above the actual default rate. Because PD feeds expected loss, provisioning and capital, the predicted PDs need to be calibrated as probabilities, not just ranks.

**Regularisation:** sklearn default (L2, C = 1.0), stated explicitly. With about 246k training rows across 12 screened features (roughly 20,500 rows per feature), the penalty has negligible effect on the coefficients, so the fit is effectively maximum likelihood. Tuning C is left to notebook 03, so the baseline stays a fixed benchmark.

**Solver:** newton-cholesky, which suits many rows and few features (246k × 12) and is deterministic, so the fit is reproducible without a seed. 

`max_iter` = 1000 sets a ceiling well above what the solver needs; convergence is confirmed in 3.2 from n_iter_. 

`random_state` = 42 is set for consistency with the train/test split but has no effect on this solver.

In [ ]:
model = LogisticRegression(solver='newton-cholesky', max_iter=1000, random_state=42, penalty='l2', class_weight=None, C=1.0)

### 3.2 Fit

**Explanation**: Intercept about -2.4: with all WOE at zero the model predicts close to the portfolio average, and ln(0.0807 / 0.9193) = -2.43. No class weighting is applied, so nothing shifts it. The intercept can move a little from that because average WOE is not exactly zero, but the average predicted PD on train should match the 8.07% default rate.

In [ ]:

model.fit(X_train, y_train)

coef = pd.Series(model.coef_[0], index=X_train.columns)

coef.sort_values()

print(model.intercept_)

print(model.n_iter_)

print(model.classes_)

In [ ]:
from statsmodels.api import Logit

sm_fit = Logit(y_train, add_constant(X_train)).fit()
sm_fit.summary()

In [ ]:
model.predict_proba(X_train)[:, 1].mean()

y_train.mean()

**Observation:** Intercept -2.4416 (sklearn) and -2.4417 (statsmodels), against the predicted -2.4 (base-rate log-odds -2.43). The solver converged in 5 iterations, far below the 1,000 ceiling. Mean predicted PD on train is 8.0731% against an actual default rate of 8.0730%. This match is guaranteed rather than lucky, because the unpenalised intercept balances predicted and actual defaults over the whole training set. It confirms the fit ran correctly but says nothing about calibration within score bands, which is tested in 4.3. The sklearn (L2, C = 1) and statsmodels (unpenalised) coefficients agree to about 2 decimal places, supporting the 3.1 claim that the penalty is negligible at this sample size.


**Decision:** Fit accepted as the baseline, with no change to settings. Proceed to the coefficient sign check in 3.3.

### 3.3 Coefficient Sign Check

**Explanation:** Risky bins carry negative WOE, so I expect every coefficient to be negative. A coefficient is a hard fail only if it is significantly positive: its 95% confidence interval lies entirely above zero. A positive coefficient whose interval includes zero is noted and investigated, not failed.

In [ ]:
coeff_check = pd.concat([sm_fit.params, sm_fit.conf_int()], axis=1)

coeff_check.columns = ["Coefficient", "Lower CI", "Upper CI"]

cols = list(coeff_check.columns)

idx1, idx2 = cols.index("Coefficient"), cols.index("Lower CI")

cols[idx1], cols[idx2] = cols[idx2], cols[idx1]

coeff_check = coeff_check[cols]

coeff_check['Spread'] = abs(coeff_check['Lower CI'] - coeff_check["Upper CI"])

coeff_check['Spread']

coeff_check.style.format('{:.4f}')

**Observation:** All 12 retained coefficients are negative and no 95% CI reaches zero, so 1.1c is met. The weakest is NAME_INCOME_TYPE at -0.0975 (95% CI -0.1825 to -0.0124), which passes but contributes least. In the first fit DAYS_BIRTH came out at +0.064 (95% CI 0.000 to 0.127, p = 0.049), a narrow but real fail, with every VIF below 2.

**Decision:** DAYS_BIRTH was dropped because it failed 1.1c, a rule set before any results were seen, so a narrow fail is still a fail. Collinearity was not the cause: every VIF was below 2, well under the threshold of 5, so the other features could not have rebuilt it. The positive sign most likely reflects a small conditional effect of age once the external scores and employment are in the model. NAME_INCOME_TYPE is kept and will be reviewed in notebook 03, where regularisation may shrink its small coefficient towards zero.

## 4. Baseline Results

### 4.1 Discrimination (AUC / Gini, KS)

**Explanation:** Expected test Gini of at least 0.45. EXT_SOURCE_3 alone gives 0.36, and the remaining features add information on top of it, though less than their single-feature Ginis because the EXT scores overlap with each other.

In [ ]:
# Get Default Probabilities

pd_test = model.predict_proba(X_test)[:, 1]

pd_test


In [ ]:
# Calculate AUC

raw_auc_test = roc_auc_score(y_test, pd_test)

raw_auc_test

In [ ]:
# Calculate Gini

gini_test = raw_auc_test * 2 - 1

gini_test

In [ ]:
y_test.sample(10)

In [ ]:
# KS Statistic

ks_test = ks_2samp(pd_test[y_test == 1], pd_test[y_test == 0])

ks_test.statistic

**Observation:** Test AUC 0.7470, Gini 0.4941. This meets the 1.1a criterion (at least 0.45) by 0.044, well clear of the roughly ±0.01 sampling noise. It is 0.134 above the best single feature (EXT_SOURCE_3, 0.36), so the other 11 features add real but limited discrimination, consistent with their overlap. KS is 0.36, below the Gini of 0.49 as expected, and in the typical range for retail credit models. No KS criterion was set in 1.1, so it is reported as supporting context rather than tested; notebook 03 may set one.

**Decision:** Criterion met. The baseline test Gini of 0.494 is the benchmark that notebook 03 must beat. Proceed to 4.2.

### 4.2 Train vs Test Gap

**Explanation:** Gap (train Gini minus test Gini) expected to be negligible and well below the 0.02 limit in 1.1b. A logistic regression on 12 WOE features with about 246k rows has very little capacity to overfit. A small positive gap is expected because the bins were fitted on train, and test Gini carries about ±0.01 sampling noise.


In [ ]:
# Get Default Probabilities

pd_train = model.predict_proba(X_train)[:, 1]

pd_train


In [ ]:
raw_auc_train = roc_auc_score(y_train, pd_train)

raw_auc_train

In [ ]:
# Gini Calc

gini_train = raw_auc_train * 2 - 1

gini_train

In [ ]:
# Gini Train/Test Gap

gini_gap = gini_train - gini_test

gini_gap

In [ ]:
# KS Statistic

ks_train = ks_2samp(pd_train[y_train == 1], pd_train[y_train == 0])

ks_train.statistic

**Observation:** Train Gini 0.4912 and test Gini 0.4941, a gap of -0.0028 (test marginally higher), inside the sampling noise and far below the 0.02 limit. Train and test KS are 0.3655 and 0.3633, and AUC 0.7456 and 0.7470. The close agreement shows no overfitting. It does not rule out leakage that is present in both splits, but a test Gini of 0.49 is plausible for these features and not suspiciously high.

**Decision:** The 1.1b criterion is met. No change to the model. Proceed to rank ordering in 4.3.

### 4.3 Rank Ordering by Score Band

**Explanation:** Sorting the test rows into 10 equal-count bands by predicted PD, I expect the actual default rate to be about 2% in decile 1 and about 25% in decile 10, rising with each band (the 1.1d criterion: zero reversals). A reversal is tested against sampling noise: a dip larger than 2 standard errors of the gap between neighbouring bands (about 95% confidence), and also present in the train deciles, is treated as real. Within noise, I tolerate and document it. Beyond noise, I diagnose on train by checking the binning of each feature across the flipped bands. If one feature's bins drive it, the feature set or binning is revisited. If none does, it is recorded as a model-form limitation for notebook 03.

In [ ]:
test_bands = pd.DataFrame({
    'pd': pd_test,
    'y': y_test.values,
    'decile': pd.qcut(pd_test, q=10, labels=range(1, 11))
})

test_bands.sample(10)

In [ ]:
# Decile analysis

# deciles = test_bands.groupby('decile').mean()

deciles = test_bands.groupby('decile').agg('mean')

deciles.style.format("{:.2%}")

**Observation:** Decile 1 defaults at 1.38% (predicted PD 1.33%) and decile 10 at 26.29% (predicted PD 25.77%), against my expectation of about 2% and 25%. The safe end separates better than expected. The actual rate rises in every band with zero reversals. The smallest step between neighbouring deciles is 0.27 pp (between deciles 3 and 4), which is under 1 standard error of the gap (about 0.33 pp), so those two bands are barely separated even though their order holds. The next smallest is 0.69 pp (deciles 4 and 5), about 2 standard errors. All other steps are well beyond noise.

**Decision:** 1.1d is met. No change to the model or features, and no train-decile check is needed because there is no reversal to test. The thin 3-to-4 step is documented, not investigated. Predicted PD and actual rate differ by up to 0.82 pp in decile 5 (predicted 5.02%, actual 4.20%), with decile 6 under-predicted by 0.62 pp. This is a calibration question, not a ranking one, and is left to the calibration step in Section 7.

## 5. Findings Summary

*(the baseline numbers, whether they met the success criteria from 1.1, and what stood out)*

## 6. Assumptions, Limitations & Data Lineage

- **Inputs:** `inputs/` exported by `default_rate_driver_scan_notebook.ipynb` (split 80/20, stratified on `TARGET`, `random_state=42`).
- **Excluded:** `CODE_GENDER` (protected attribute). No proxy check run yet.
- **Known gaps carried over:** see `metadata['known_gaps_for_future_tuning']`.
- *(add any new assumptions made in this notebook)*

## 7. Next Steps

- [ ] Calibration - predicted PD vs actual default rate by band
- [ ] Stability - PSI between train and test score distributions
- [ ] Scorecard scaling - base score, base odds, PDO
- [ ] Gender proxy check on the retained features
- [ ] *(add from findings)*

## 8. Save Model to Registry

In [ ]:
# Folder for this model inside the registry
model_dir = os.path.join(repo_root, 'Projects', 'Flagship PD Model Project',
                         '05_Applicant_PD_Dashboard', 'registry', 'logreg_baseline_v1')

# Create it if it doesn't exist yet
os.makedirs(model_dir, exist_ok=True)

# Save the fitted model as a file inside that folder
joblib.dump(model, os.path.join(model_dir, 'model.joblib'))

shutil.copy(os.path.join(INPUT_DIR, 'binning_process.pkl'),
            os.path.join(model_dir, 'binning_process.pkl'))

In [ ]:
model_meta = {
    'name': 'logreg_baseline_v1',
    'version': '1.0',
    'role': 'champion',
    'features': list(baseline_features),
    'gini_test': float(gini_test),
    'gini_train': float(gini_train),
    'ks_test': float(ks_test.statistic),
    'sklearn_version': sklearn.__version__,
}

# Save next to the model file
with open(os.path.join(model_dir, 'metadata.json'), 'w') as fh:
    json.dump(model_meta, fh, indent=2)